# Make analysis figures

Notebook interface for the scenario-analysis workflow. It mirrors `run_analysis_figures.py` and calls the current analysis modules, including the solar-share scatter diagnostic and Analysis 1/2 tables.

In [ ]:
import sys
from pathlib import Path

# Assumes this notebook is stored in notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.common.plot_utils import apply_plot_style
from src.common.config import ANALYSIS_OUTPUT_DIR

from src.analysis.boxplots import make_figure3a
from src.analysis.scatter_plots import make_figure3b, make_storage_solar_scatter
from src.analysis.stacked_bar import make_stacked_bar

from src.analysis.flex_metrics import compute_flex_ratios_all, make_analysis3_table
from src.analysis.flex_plots import make_figure4a, make_figure4b
from src.analysis.source_data import export_figure4_source_data

from src.analysis.analysis1_indicators import make_analysis1_tables
from src.analysis.analysis2_sensitivity import make_analysis2_sensitivity

apply_plot_style()
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Analysis output directory:", ANALYSIS_OUTPUT_DIR)

## Run settings

In [ ]:
save_png = True
save_pdf = False
save_svg = False
export_source_data = True

# Optional: set to False if statsmodels is not installed
add_quantile_fit = False

## Figure 3a: storage capacity boxplots

In [ ]:
fig3a, data3a = make_figure3a(
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
)

data3a.head()

## Figure 3b: storage intensity vs VRE share

In [ ]:
fig3b, data3b = make_figure3b(
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
)

data3b.head()

## Additional scatter: storage intensity vs solar share

In [ ]:
fig_solar, data_solar = make_storage_solar_scatter(
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
    add_pypsa=True,
    add_quantile_fit=add_quantile_fit,
    save_csv=True,
)

# data_solar is a dictionary in the updated interface
data_solar.keys() if isinstance(data_solar, dict) else type(data_solar)

## Stacked power-capacity bars

In [ ]:
fig_stack, data_stack = make_stacked_bar(
    years=(2030, 2050),
    as_share=False,
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
)

data_stack.head()

## Flexibility indices and plots

In [ ]:
indices = compute_flex_ratios_all()
indices.head()

In [ ]:
fig4a, data4a = make_figure4a(
    indices=indices,
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
)

data4a.head()

In [ ]:
fig4b, data4b = make_figure4b(
    indices=indices,
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
)

data4b.head()

## Analysis 1: IAM spread and benchmark indicators

In [ ]:
analysis1_iam_spread, analysis1_benchmarks, analysis1_inset = make_analysis1_tables(
    save_csv=True,
)

analysis1_inset

## Analysis 2: within-model scenario sensitivity

In [ ]:
fig_analysis2, data_analysis2 = make_analysis2_sensitivity(
    year=2050,
    plot_mode="range",
    save_png=save_png,
    save_pdf=save_pdf,
    save_svg=save_svg,
    save_csv=True,
)

data_analysis2["sensitivity_table"].head()

## Analysis 3 table

In [ ]:
analysis3_df, analysis3_corr = make_analysis3_table(
    save_csv=True,
    include_benchmarks=False,
)

analysis3_df.head()

## Export consolidated source data

In [ ]:
if export_source_data:
    export_figure4_source_data(
        figure3a=data3a,
        figure3b=data3b,
        figure4a=data4a,
        figure4b=data4b,
    )
    print("Consolidated source data exported.")

print("Workflow complete.")

In [ ]:
import matplotlib.pyplot as plt
plt.show()